# Lesson 3.2 — Custom Slash Commands & Skills (context:fork, allowed-tools)

**Level: Intermediate**

## Learning Objectives

By the end of this lesson, you will be able to:

- Place project-scoped commands in .claude/commands/ vs user-scoped in ~/.claude/commands/
- Author a SKILL.md with context:fork, allowed-tools, and argument-hint frontmatter
- Use context:fork to isolate verbose/exploratory skills from the main session
- Choose skills (on-demand) vs CLAUDE.md (always-loaded)

**You will produce:** A project /review slash command and a forked, tool-restricted SKILL.md.

## Concepts

**Slash commands:** project-scoped in `.claude/commands/` (shared via VCS, available after clone/pull) vs user-scoped in `~/.claude/commands/` (personal). Sample Question 4: a team-wide `/review` command belongs in `.claude/commands/`.

**Skills:** live in `.claude/skills/` as `SKILL.md` files with frontmatter:
- **`context: fork`** runs the skill in an isolated sub-agent context so verbose output does not pollute the main conversation.
- **`allowed-tools`** restricts which tools the skill may use (e.g. limit to file writes to prevent destructive actions).
- **`argument-hint`** prompts the developer for required parameters when invoked without args.
- Personal variants go in `~/.claude/skills/` with different names to avoid affecting teammates.

**Skills vs CLAUDE.md:** skills are on-demand, task-specific workflows; CLAUDE.md is always-loaded universal standards.

> **Reference:** [Anthropic - Claude Code skills & commands](https://docs.anthropic.com/en/docs/claude-code/skills)

In [ ]:
import os, pathlib
ROOT = pathlib.Path('cc_project'); ROOT.mkdir(exist_ok=True)
os.chdir(ROOT)
(pathlib.Path('.claude') / 'rules').mkdir(parents=True, exist_ok=True)
(pathlib.Path('.claude') / 'commands').mkdir(parents=True, exist_ok=True)
(pathlib.Path('.claude') / 'skills').mkdir(parents=True, exist_ok=True)
print('project root:', os.getcwd())

## Code-Along: Step 1 — A project-scoped /review command

Stored in .claude/commands/ so every developer gets it from version control.

In [ ]:
review_cmd = '''---
description: Run the team code-review checklist
argument-hint: <path-or-PR>
---
Review $ARGUMENTS against our checklist:
1. Bugs and security issues (report)
2. Minor style/local patterns (skip)
3. Output: location, issue, severity, suggested fix
'''
(pathlib.Path('.claude/commands') / 'review.md').write_text(review_cmd, encoding='utf-8')
print(review_cmd)

## Code-Along: Step 2 — A forked, tool-restricted SKILL.md

`context: fork` isolates verbose codebase analysis; `allowed-tools` limits it to safe operations; `argument-hint` prompts for a target.

In [ ]:
skill = '''---
name: analyze-codebase
description: Produce a structure map of a package (verbose)
context: fork
allowed-tools: Read, Grep, Glob
argument-hint: <package-path>
---
Map the structure of $ARGUMENTS: entry points, modules, and key dependencies.
Return a concise summary to the main session.
'''
skdir = pathlib.Path('.claude/skills/analyze-codebase'); skdir.mkdir(parents=True, exist_ok=True)
(skdir / 'SKILL.md').write_text(skill, encoding='utf-8')
print(skill)

## Code-Along: Step 3 — Skill vs CLAUDE.md decision

Always-loaded universal standards -> CLAUDE.md; on-demand task workflow -> skill.

In [ ]:
def placement(need):
    return 'CLAUDE.md' if need['always_loaded'] and need['universal'] else 'skill'

print('coding standards:', placement({'always_loaded': True, 'universal': True}))
print('brainstorm alternatives:', placement({'always_loaded': False, 'universal': False}))

## Exercise

Create a `generate-migration` skill restricted to file-write tools only (no Bash), with an `argument-hint` for the table name and `context: fork`. Then assert the SKILL.md frontmatter does NOT grant Bash, proving the allowed-tools restriction prevents destructive actions.

In [ ]:
# TODO: write .claude/skills/generate-migration/SKILL.md with allowed-tools: Write, Edit
# TODO: assert 'Bash' not in the frontmatter allowed-tools line

## Key Takeaways

- Project /commands are shared via .claude/commands/; user commands live in ~/.claude/commands/.
- SKILL.md frontmatter supports context:fork, allowed-tools, and argument-hint.
- context:fork isolates verbose/exploratory skills from the main conversation.
- Skills are on-demand; CLAUDE.md is always-loaded universal standards.

---

**Next:** [Lesson 3.3 — Path-Specific Rules](./lesson-3.3-path-specific-rules.ipynb)